# 01 — Web Scraping: IMDb User Reviews & RogerEbert Professional Reviews

This notebook collects the raw data used throughout the project. We scrape two sources:

1. **IMDb** — general audience reviews across five genres (Adventure, Comedy, Crime, Horror, Superhero)
2. **RogerEbert.com** — professional critic reviews matched to the IMDb movies

## Scraping challenges addressed
- **IMDb 403 errors:** resolved by adding a User-Agent header
- **Spoiler-hidden reviews on IMDb:** reviews behind a "spoiler warning" were invisible via `requests` alone. Solved with a Selenium WebDriver that clicks the expander icon and waits for the review text to load
- **Different rating scales:** RogerEbert uses a 0–4 star scale encoded in the image class name; IMDb uses 0–10. We extract the star rating via regex on the image class and rescale to 10 for direct comparison
- **Session persistence on RogerEbert:** an earlier bug caused empty results when switching genres; fixed by using a single `requests.Session()`

## Team contribution note
This scraping pipeline was primarily built by Evan Carlson (IMDb scraper, full pipeline) and Elpidio Azarcon (RogerEbert review text scraping). I (Roy Ho) contributed the RogerEbert rating extraction logic — identifying that the star image's class name contained the rating, writing the XPath + regex to pull it out, and the 0–40 to 0–10 rescaling.

The cleaned output of this notebook is saved under `data/user_reviews/` and `data/professional_reviews/`.

## Setup

In [ ]:
import requests
import lxml.html as lx
import re
import time
import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

session = requests.Session()
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/134.0.0.0 Safari/537.36"
    )
}
IMDB_URL = "https://www.imdb.com"


## Part 1 — IMDb scraping

### Step 1: Get trending movies for a genre

IMDb's "By Genre" pages each include a "Popular Movies" carousel. Even though it's a horizontal scroller, all links are rendered in the same HTML container, so we can extract them without Selenium.

In [ ]:
def get_movies(genre_link):
    """Return a deduplicated list of movie URLs from a genre page."""
    result = requests.get(genre_link, headers=headers)
    tree = lx.fromstring(result.content)
    links = tree.xpath(
        "(//div[@data-testid='shoveler-items-container'])[1]"
        "//a[contains(@href, '/title/')]"
    )

    seen = set()
    cleaned_links = []
    for link in links:
        href = str(link.get("href"))
        # Dedupe on the tail of the title ID
        if href[-2:] not in seen:
            cleaned_links.append(href.split('?')[0])
            seen.add(href[-2:])
    return cleaned_links


# Genre pages used in the original run
genre_pages = {
    "adventure": "https://www.imdb.com/interest/in0000006/",
    "comedy":    "https://www.imdb.com/interest/in0000034/",
    "crime":     "https://www.imdb.com/interest/in0000040/",
    "horror":    "https://www.imdb.com/interest/in0000112/",
    "superhero": "https://www.imdb.com/interest/in0000008/",
}


### Step 2: Scrape reviews for a single movie

This is where the spoiler-warning problem appears. For each review link, we load the page in Selenium, try to click the expander icon, wait for the text to render, then pull it. If there's no expander, we fall back to reading the text directly.

In [ ]:
def get_reviews(movie_link):
    """Return a list of (user_rating, review_text) tuples for one movie."""
    reviews = []

    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options,
    )

    response = requests.get(
        IMDB_URL + movie_link + "reviews/", headers=headers
    )
    tree = lx.fromstring(response.content)
    links = tree.xpath("//a[@class='ipc-title-link-wrapper']")
    time.sleep(1)

    for link in links:
        driver.get(IMDB_URL + link.get("href"))
        try:
            show_more = driver.find_element(By.CLASS_NAME, "expander-icon-wrapper")
            show_more.click()
            time.sleep(1)
        except Exception:
            # No spoiler warning on this review; proceed without clicking
            pass

        review_els = driver.find_elements("xpath", "//div[@class='text show-more__control']")
        rating_els = driver.find_elements("xpath", "//span[@class='rating-other-user-rating']//span")

        individual_review = [el.text for el in review_els]
        try:
            reviews.append((rating_els[0].text, individual_review))
        except Exception:
            # Review exists but user left no star rating
            continue
        time.sleep(1)

    driver.quit()
    return reviews


### Step 3: Text preprocessing

Lowercase, strip newlines and special characters, tokenize, remove English stop words. This gives the downstream sentiment models cleaner input.

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def process_reviews(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    return " ".join(w for w in words if w not in STOP_WORDS)


### Step 4: Combine everything into a per-movie scraper

In [ ]:
def get_movie_data(movie_link):
    """Return (title, aggregate_rating, [(user_score, processed_review), ...])."""
    result = requests.get(IMDB_URL + movie_link, headers=headers)
    tree = lx.fromstring(result.content)

    title = tree.xpath("//title[1]")[0].text
    rating = tree.xpath(
        "//div[@data-testid='hero-rating-bar__aggregate-rating']//span"
    )[1].text

    raw_reviews = get_reviews(movie_link)
    processed = []
    for score, review_chunks in raw_reviews:
        if not review_chunks:
            continue
        processed.append((score, process_reviews(review_chunks[0])))

    return (title, rating, processed)


def create_genre_dataframe(genre_data):
    """Flatten per-movie scraped data into a tidy DataFrame."""
    rows = []
    for title, rating, review_list in genre_data:
        for score, review in review_list:
            rows.append({
                "Title": title,
                "Rating": rating,
                "Review Score": score,
                "Review Text": review,
            })
    return pd.DataFrame(rows)


### Step 5: Run the full scrape

Each genre took roughly 40 minutes because of the Selenium clicks and required `time.sleep` delays. In practice this was run once and saved to disk.

In [ ]:
# Example run (commented out; uncomment to re-scrape)
#
# for genre, page_url in genre_pages.items():
#     movies = get_movies(page_url)
#     genre_data = [get_movie_data(m) for m in movies]
#     df = create_genre_dataframe(genre_data)
#     df.to_csv(f"../data/user_reviews/{genre}.csv", index=False)
#     print(f"{genre}: {len(df)} reviews saved")


## Part 2 — RogerEbert professional reviews

For each scraped IMDb movie, search RogerEbert.com's review endpoint for a match, then pull the review text and rating.

The trickiest piece was the rating: RogerEbert encodes the star count in the image's CSS class (`h-7 filled starXX`, where `XX` runs 5, 10, 15, ..., 40 for 0.5-star increments up to 4 stars). We pull the class string, regex out the number, then rescale 0–40 → 0–10 to match IMDb.

In [ ]:
def get_professional_reviews(movie_name):
    """Return (title, rating_out_of_10, processed_review_text) or None."""
    # Strip "(YYYY) - IMDb" suffix from the IMDb page title
    m = re.match(r"^(.*?)\s*\(\d{4}\)\s*-\s*IMDb", movie_name)
    movie_name_cleaned = m.group(1) if m else movie_name

    result = session.get(
        "https://www.rogerebert.com/reviews?_search_reviews",
        headers=headers,
        params={"_search_reviews": movie_name_cleaned},
    )
    tree = lx.fromstring(result.content)

    link = tree.xpath(
        "//a[@class='image-hover cursor-pointer relative rounded "
        "flex flex-col justify-end'][1]"
    )
    if not link:
        return None

    time.sleep(1)
    response = session.get(link[0].get("href"), headers=headers)
    tree1 = lx.fromstring(response.content)

    review_text = tree1.xpath('//article//p/text()')
    full_review = " ".join(review_text)

    # Rating is encoded in the class of the stars image
    star_image = tree1.xpath("//img[contains(@class, 'h-7 filled')]//@class")
    try:
        match = re.search(r"star(\d+)", star_image[0])
        rating = int(match.group(1))
        rating = (rating / 40) * 10  # rescale 0–40 to 0–10
        return (movie_name, rating, process_reviews(full_review))
    except Exception:
        return None


def remove_duplicate_movies(movie_list):
    seen, out = set(), []
    for m in movie_list:
        if m not in seen:
            out.append(m); seen.add(m)
    return out


def rogerebert_review_machine(movie_df):
    titles = remove_duplicate_movies(movie_df["Title"].to_list())
    return [get_professional_reviews(t) for t in titles]


In [ ]:
# Example run (commented out)
#
# for genre in ["adventure", "comedy", "crime", "horror", "superhero"]:
#     user_df = pd.read_csv(f"../data/user_reviews/{genre}.csv")
#     pro_results = rogerebert_review_machine(user_df)
#     pro_results = [r for r in pro_results if r is not None]
#     pro_df = pd.DataFrame(pro_results, columns=["Title", "Pro Rating", "Review"])
#     pro_df.to_csv(f"../data/professional_reviews/{genre}.csv", index=False)
#     print(f"{genre}: {len(pro_df)} professional reviews saved")


## Output

After running this notebook end-to-end, the `data/` directory contains:

```
data/
├── user_reviews/
│   ├── adventure.csv     (710 reviews)
│   ├── comedy.csv        (708 reviews)
│   ├── crime.csv         (668 reviews)
│   ├── horror.csv        (700 reviews)
│   └── superhero.csv     (702 reviews)
└── professional_reviews/
    ├── adventure.csv     (30 reviews)
    ├── comedy.csv        (30 reviews)
    ├── crime.csv         (29 reviews)
    ├── horror.csv        (29 reviews)
    └── superhero.csv     (30 reviews)
```

Total: ~3,488 user reviews, ~148 professional reviews.